In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error

In [ ]:
orders = pd.read_csv("olist_orders_dataset.csv",
                     parse_dates=["order_purchase_timestamp",
                                  "order_delivered_customer_date"])
items = pd.read_csv("olist_order_items_dataset.csv")
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=orders)

https://docs.google.com/spreadsheets/d/1vIsn6t1vJStJpcPXyHKwY9Xj_x-QGp8R8lUW2k9QiqE/edit#gid=0


In [ ]:
orders = orders.dropna(subset=["order_delivered_customer_date"])
orders["delivery_days"] = (orders["order_delivered_customer_date"]
                           - orders["order_purchase_timestamp"]).dt.days
orders["delivery_days"].describe()

/tmp/ipykernel_1124/3086251318.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders["delivery_days"] = (orders["order_delivered_customer_date"]


,delivery_days
count,96476.000000
mean,12.094086
std,9.551746
min,0.000000
25%,6.000000
50%,10.000000
75%,15.000000
max,209.000000


In [ ]:
totals = items.groupby("order_id").agg(price=("price", "sum"),
                                       freight=("freight_value", "sum"),
                                       n_items=("order_item_id", "count")).reset_index()
df = orders.merge(totals, on="order_id")
df["month"] = df["order_purchase_timestamp"].dt.month
df["weekday"] = df["order_purchase_timestamp"].dt.dayofweek
df = df[df["delivery_days"] <= 60]  # remove extreme outliers

X = df[["price", "freight", "n_items", "month", "weekday"]]
y = df["delivery_days"]
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,price,freight,n_items,month,weekday
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,8,29.99,8.72,1,10,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,13,118.70,22.76,1,7,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,9,159.90,19.22,1,8,2
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,13,45.00,27.20,1,11,5
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,2,19.90,8.72,1,2,1


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
lr = LinearRegression().fit(X_train, y_train)
pred = lr.predict(X_test)
print(f"Linear Regression: R2 = {r2_score(y_test, pred):.3f}, MAE = {mean_absolute_error(y_test, pred):.2f} days")

Linear Regression: R2 = 0.028, MAE = 5.94 days


In [ ]:
dt = DecisionTreeRegressor(max_depth=8, random_state=42).fit(X_train, y_train)
pred = dt.predict(X_test)
print(f"Decision Tree: R2 = {r2_score(y_test, pred):.3f}, MAE = {mean_absolute_error(y_test, pred):.2f} days")

Decision Tree: R2 = 0.230, MAE = 5.15 days


In [ ]:
knn = KNeighborsRegressor(n_neighbors=10).fit(X_train_s, y_train)
pred = knn.predict(X_test_s)
print(f"KNN: R2 = {r2_score(y_test, pred):.3f}, MAE = {mean_absolute_error(y_test, pred):.2f} days")

KNN: R2 = 0.177, MAE = 5.37 days
